# QC filtering 00 — Cohort and frozen Phase 1 parameters

Does Pal et al.'s per-sample QC remove cells at different rates in BRCA1+/- donors
and premenopausal WT donors? The donor is the unit of replication. This series
answers that in two phases. Phase 1 does not transfer labels.

| Notebook | Question |
|---|---|
| `00` cohort | Which 12 donors are in the comparison, and which analysis choices are frozen before any test? |
| `01` filtering rate | Does the removed fraction, overall and by reason, differ by genotype? |
| `02` Pal reference | Do the authors' own labels reproduce their retained-cell result (P = 0.14)? |
| `03` tipping point | How LP-enriched would BRCA1 removed cells have to be to hide an LP expansion? |

**This notebook does not test genotype.** It selects the cohort, writes one row
per cell and one row per donor, and freezes the Phase 1 parameters. The uniform
thresholds and the depth target are computed here because they are properties of
the cohort, not results of the comparison.

The cohort is every BRCA1 pre-neoplastic Total sample, including the two
post-oophorectomy donors, and the premenopausal Normal Total samples only.
Menopause is filtered for Normal samples and is not filtered for BRCA1.

In [ ]:
import logging

import pandas as pd
from IPython.display import Markdown, display

from signals_in_the_noise.analysis.qc_filtering import (
    brca1_wt_cohort,
    output_directory,
    write_phase1_outputs,
)
from signals_in_the_noise.preprocessing.gse161529 import GSE161529
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

OUTPUT_DIR = output_directory()
OUTPUT_DIR

## 1. Cohort

Specimens are keyed by GEO file name. Donor ids such as `N-0093` come from the
patient in the sample title, which is not always the number in the file name
(`N-PM0095` is patient 0093).

In [ ]:
gse = GSE161529()
cohort = brca1_wt_cohort(gse.objects.values())
cohort.overview()

## 2. Cell table, donor table, and frozen parameters

`cells.csv` has one row per barcode. A cell that fails several filters keeps
all four flags and also gets one exclusive reason, in the order high mitochondrial
fraction, low genes, high genes, high library size. Retained cells have no
exclusive reason.

`donors.csv` reuses the specimen covariate table and adds genotype, parity,
the inferred source prefix, the published GEO counts, and the median UMI and
gene counts over all barcodes.

`analysis_params.json` stores the Phase 1 section. Re-running this notebook
does not remove a Phase 2 section. The near-threshold constant `c` is left
empty for notebook 02.

The handoff below is the text to paste back after this notebook runs.

In [ ]:
handoff = write_phase1_outputs(cohort, OUTPUT_DIR)
donors = pd.read_csv(OUTPUT_DIR / "donors.csv")
display(
    donors[
        [
            "donor",
            "genotype",
            "menopause_status",
            "parity",
            "source_prefix",
            "n_cells",
            "n_retained",
            "n_removed",
            "removed_fraction",
            "median_total_counts",
        ]
    ]
)
display(Markdown(handoff))